# 07 — Smart Execute: One Function Call from Question to Answer

**The pitch.** You have a question. You don't want to learn 88 cognitive patterns (16 free + 72 enterprise), chain orchestration, or prompt engineering. You want a great answer. `smart_execute(question)` routes your question through mycontext's intelligence layer — picks the right tier, selects the right template, assembles an expert-grade prompt, calls the model — and hands you the response plus a full audit trail.

**Who this is for:** Anyone who wants the fastest path from question to quality answer.

**What you'll learn:**
- How `smart_execute` routes simple / moderate / complex questions to three different tiers
- How to inspect the `meta` dict to see *exactly* what happened
- How `smart_prompt` lets you preview the assembled prompt before spending tokens
- How `smart_generic_prompt` gives you a fast, no-compile-cost prompt for any template
- How **web Output Style** maps to API `quality` — and what to use in the SDK instead

**Visual map** → [assets/overview.html](assets/overview.html)

**Next:** **08** = Prompt Architect (build / parse / improve). **09** = Template selection intelligence. **10** = Quality gates.

## Research grounding

> **Routing model:** Three-tier complexity routing is grounded in *Khot et al. 2022 — "Decomposed Prompting"* — routing complex questions through specialist reasoning templates produces measurably more structured outputs than single-pass prompting. The **CAI benchmark** (mycontext's own evaluation, 200+ test cases) shows template-driven prompts produce 1.8–2.5x better outputs on analytical questions, with no benefit on simple factual queries. Three tiers exist to apply that lift only where it's warranted — keeping simple queries fast and cheap.

> **Thinking strategies in templates:** Where templates inject a reasoning strategy, they map directly to published techniques: Chain of Thought *(Wei et al. 2022)*, Tree of Thought *(Yao et al. 2023)*, Self-Reflection / Reflexion *(Shinn et al. 2023)*.

## The problem this solves

Every AI framework makes you make decisions you shouldn't have to make:

| Decision | Without mycontext | With `smart_execute` |
|----------|-------------------|----------------------|
| Which prompt template? | Browse docs, guess | Automatic |
| Simple or complex routing? | Manual triage | Three-tier auto-routing |
| Prompt quality before call? | Hope for the best | Built-in |
| Which model settings? | Tune per-call | Sensible defaults |

`smart_execute` is the **zero-decision path**. One call. We decide.

## Install and setup

In [1]:
# %pip install -q -U "mycontext-ai>=0.11.1"
import os
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

from dotenv import load_dotenv
from IPython.display import display_markdown

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

def md(s):
    display_markdown(s, raw=True)

import mycontext

md(f"**mycontext-ai** `{getattr(mycontext, '__version__', '?')}`")


**mycontext-ai** `0.11.0`

## The three tiers — how routing works

Before the first call, it helps to know what's happening under the hood:

| Tier | Triggered when | What happens |
|------|---------------|---------------|
| **Raw** | Simple, factual, or conversational | Sends a clean direct prompt — no template overhead |
| **Single template** | Analytical, structured reasoning needed | Picks the best cognitive pattern from the catalog of 88 patterns, builds a full Context |
| **Integrated pipeline** | Multi-angle, complex question | Chains 2–5 templates, integrates them into one unified prompt |

The routing happens in `assess_complexity()` — heuristic-first, LLM-fallback.

In code, `assess_complexity()` returns a **`ComplexityResult`**: use **`recommendation`** (`raw` | `single_template` | `integrated`) for the three execution paths, and **`complexity`** (`low` | `medium` | `high`) for signal strength. There is no `.tier` attribute.

## Step 1 — Simple question (raw tier)

In [2]:
from mycontext.intelligence import smart_execute, assess_complexity

simple_q = "What does HTTP 429 mean?"

# Check complexity first (heuristic, no API key needed)
complexity = assess_complexity(simple_q, skip_heuristic=False)
md(f"Question: {simple_q!r}")
md(f"**Complexity:** `{complexity.complexity}`  ·  **Recommendation:** `{complexity.recommendation}`")
md(f"**Reasoning:** {complexity.reasoning}")


Question: 'What does HTTP 429 mean?'

**Complexity:** `low`  ·  **Recommendation:** `raw`

**Reasoning:** Short question (7 tokens) with no multi-domain complexity signals — heuristic pre-screen classified as raw.

In [ ]:
import json
response, meta = smart_execute(simple_q, provider="openai")
md("=== **Response** ===")
md((str(response)[:500-3] + '...' if len(str(response)) > 500 else str(response)))
md("\n=== Meta ===")
_sj = json.dumps(meta, indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")


=== Response ===

HTTP 429 is a status code that indicates "Too Many Requests." This response is sent by a server when a user has sent too many requests in a given amount of time, exceeding the rate limit set by the server. It serves as a way to protect the server from being overwhelmed by too many requests, which could lead to performance issues or downtime. When a client receives a 429 status, it is typically advised to slow down the frequency of requests and may also include a "Retry-After" header indicatin...


=== Meta ===

```json
{
  "assessment": {
    "complexity": "low",
    "domains": [],
    "reasoning_type": "explanatory",
    "recommendation": "raw",
    "reasoning": "Short question (7 tokens) with no multi-domain complexity signals \u2014 heuristic pre-screen classified as raw.",
    "best_template": null
  },
  "mode": "raw",
  "templates_used": []
}
```

## Step 2 — Moderate question (single template tier)

In [4]:
moderate_q = "What are the root causes of high customer churn in a B2B SaaS product?"

complexity = assess_complexity(moderate_q, skip_heuristic=False)
md(f"Question: {moderate_q!r}")
md(f"**Complexity:** `{complexity.complexity}`  ·  **Recommendation:** `{complexity.recommendation}`")
md(f"**Reasoning:** {complexity.reasoning}")


Question: 'What are the root causes of high customer churn in a B2B SaaS product?'

**Complexity:** `medium`  ·  **Recommendation:** `single_template`

**Reasoning:** The question involves identifying the root causes of a specific issue (customer churn) within a business context, which benefits from a structured analytical framework.

In [5]:
import json
response, meta = smart_execute(moderate_q, provider="openai")
md("=== Response (first 800 chars) ===")
md((str(response)[:800-3] + '...' if len(str(response)) > 800 else str(response)))
md("\n=== Meta — notice: `mode` / `assessment.recommendation` + `templates_used` ===")
_sj = json.dumps(meta, indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")


=== Response (first 800 chars) ===

### 1. PROBLEM DEFINITION
- **Problem statement**: High customer churn rates in our B2B SaaS product exceed industry benchmarks, leading to revenue and customer relationship instability.
- **Observable symptoms**: 
  - Customer cancellation rates increased by 25% over the past six months.
  - Decrease in customer engagement metrics, such as login frequency and feature utilization.
- **Impact**: 
  - Revenue loss estimated at $2 million annually due to churn.
  - Affected stakeholders: Sales team, Customer Success team, and Product Development team.
- **When started**: Notable increase in churn began approximately six months ago.

### 2. SYMPTOM VS. CAUSE
**Symptoms (observable effects)**:
- **Symptom 1**: Increased number of customer support inquiries related to product usability.
- **S...


=== Meta — notice: `mode` / `assessment.recommendation` + `templates_used` ===

```json
{
  "assessment": {
    "complexity": "medium",
    "domains": [
      "business",
      "organizational"
    ],
    "reasoning_type": "diagnostic",
    "recommendation": "single_template",
    "reasoning": "The question involves identifying the root causes of a specific issue (customer churn) within a business context, which benefits from a structured analytical framework.",
    "best_template": "root_cause_analyzer"
  },
  "mode": "single",
  "templates_used": [
    "root_cause_analyzer"
  ]
}
```

## Step 3 — Complex question (integrated pipeline tier)

In [6]:
complex_q = (
    "We're considering acquiring a Series B fintech company that does embedded lending. "
    "Analyze the strategic fit, key risks, and what our integration roadmap should look like "
    "for a board presentation."
)

complexity = assess_complexity(complex_q, skip_heuristic=False)
md(f"**Complexity:** `{complexity.complexity}`  ·  **Recommendation:** `{complexity.recommendation}`")
md(f"**Reasoning:** {complexity.reasoning}")


**Complexity:** `high`  ·  **Recommendation:** `integrated`

**Reasoning:** The question involves multiple domains including business strategy, technical integration, and organizational risks, making it complex and requiring a comprehensive approach. An integrated framework is necessary to address the strategic fit, risks, and integration roadmap effectively.

In [7]:
import json
response, meta = smart_execute(complex_q, provider="openai")
md("=== Response (first 1000 chars) ===")
md((str(response)[:1000-3] + '...' if len(str(response)) > 1000 else str(response)))
md("\n=== Meta — notice: templates_used is now a list ===")
_sj = json.dumps(meta, indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")


=== Response (first 1000 chars) ===

### Strategic Fit Analysis: SWOT Analysis

**Strengths:**
- **Innovative Technology:** The Series B fintech company has developed advanced embedded lending technology, enhancing customer experience and operational efficiency.
- **Market Position:** The company holds a strong position in a growing segment, which aligns with our strategic goal of expanding into fintech solutions.
- **Synergy Potential:** There are significant opportunities for cross-selling and integration with our existing financial services.

**Weaknesses:**
- **Integration Complexity:** The existing technology stack and operational processes may be incompatible, creating potential hurdles during integration.
- **Cultural Misalignment:** Differences in corporate culture and values may pose challenges in merging teams and aligning objectives.
- **Limited Brand Recognition:** The fintech brand may not be well-known to our customer base, which could impact adoption rates.

**Opportunities:**
- **Market Expansion:** Acq...


=== Meta — notice: templates_used is now a list ===

```json
{
  "assessment": {
    "complexity": "high",
    "domains": [
      "business",
      "technical",
      "organizational"
    ],
    "reasoning_type": "strategic",
    "recommendation": "integrated",
    "reasoning": "The question involves multiple domains including business strategy, technical integration, and organizational risks, making it complex and requiring a comprehensive approach. An integrated framework is necessary to address the strategic fit, risks, and integration roadmap effectively.",
    "best_template": null
  },
  "mode": "multi",
  "templates_used": [
    "swot_analyzer",
    "gap_analyzer"
  ]
}
```

## Step 4 — `smart_prompt`: see the prompt before you pay for it

`smart_execute` runs the full call. Sometimes you want to **inspect the assembled prompt first** — check its quality, tweak parameters, or use it in your own pipeline. That's `smart_prompt`.

In [8]:
from mycontext.intelligence import smart_prompt

# Returns ComposedPrompt — no LLM call yet
composed = smart_prompt(
    "What are the risks of deploying ML models without monitoring?",
    provider="openai",
)

prompt_text = composed.to_string()
md("=== Assembled prompt (first 800 chars) ===")
md((str(prompt_text)[:800-3] + '...' if len(str(prompt_text)) > 800 else str(prompt_text)))
md(f"\nTotal length: {len(prompt_text)} chars")


=== Assembled prompt (first 800 chars) ===

Conduct a detailed risk assessment on the question: "What are the risks of deploying ML models without monitoring?" Your analysis should be structured into the following seven sections:

1. **SITUATION OVERVIEW**: Provide the context and objectives of deploying ML models, identify the stakeholders affected, outline the time horizon, and define the success criteria.

2. **RISK IDENTIFICATION**: Identify both obvious and hidden risks across five categories: Strategic (market changes, competitive threats), Operational (process failures, resource constraints), Financial (cost overruns, revenue shortfalls), Compliance/Legal (regulatory changes, legal liabilities), and External (economic factors, political/social changes).

3. **RISK ANALYSIS**: For each identified risk, describe what could g...


Total length: 1893 chars

In [9]:
import json
# ComposedPrompt can export to any format
msgs = composed.to_messages()
md("OpenAI messages format:")
_sj = json.dumps(msgs, indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")


OpenAI messages format:

```json
[
  {
    "role": "user",
    "content": "Conduct a detailed risk assessment on the question: \"What are the risks of deploying ML models without monitoring?\" Your analysis should be structured into the following seven sections:\n\n1. **SITUATION OVERVIEW**: Provide the context and objectives of deploying ML models, identify the stakeholders affected, outline the time horizon, and define the success criteria.\n\n2. **RISK IDENTIFICATION**: Identify both obvious and hidden risks across five categories: Strategic (market changes, competitive threats), Operational (process failures, resource constraints), Financial (cost overruns, revenue shortfalls), Compliance/Legal (regulatory changes, legal liabilities), and External (economic factors, political/social changes).\n\n3. **RISK ANALYSIS**: For each identified risk, describe what could go wrong, assign a probability (1\u20135) and impact (1\u20135) score, calculate the risk score (P\u00d7I), identify triggers, and specify the timeframe for when this risk could occur.\n\n4. **RISK PRIORITIZATION**: Classify risks as Critical (score 15\u201325), Significant (score 8\u201314), or Minor (score 1\u20137), indicating the need for immediate action, active management, or monitoring.\n\n5. **MITIGATION STRATEGIES**: For each major risk, propose strategies to avoid, reduce, transfer, accept, or create contingency plans.\n\n6. **RISK-BENEFIT ANALYSIS**: Assess the expected value of the decision, evaluate risk tolerance, and explore alternative approaches with different risk profiles.\n\n7. **RISK SUMMARY**: Highlight the top 3\u20135 risks needing immediate attention, state the overall risk level (Low / Medium / High), recommend a risk management approach, and provide a GO / NO-GO recommendation with supporting reasoning.\n\nEnsure to objectively assess probability and impact, consider cascading and compounding risks, balance risk aversion with opportunity, and incorporate evidence and historical data when available."
  }
]
```

In [10]:
# Score the prompt BEFORE calling the LLM
from mycontext.intelligence import QualityMetrics

if hasattr(composed, 'context') and composed.context is not None:
    qm = QualityMetrics(mode="heuristic")
    score = qm.evaluate(composed.context)
    md(f"Prompt quality score: {score.overall * 100:.0f}/100")
    for _dk, _dv in score.dimensions.items():
        md(f"  {_dk.name.title()}: {_dv * 100.0:.0f}")
else:
    md("Quality scoring available when context is attached to ComposedPrompt")


Quality scoring available when context is attached to ComposedPrompt

## Web app vs SDK: tuning answer shape (0.11+)

In the **web app**, the Smart Execute panel includes **Output Style** controls (verbosity, answer first, self-verify). Those map to API **`quality`** overrides on the execution request.

The SDK **`smart_execute()`** path does **not** expose that same `quality` payload. To steer output in code, use the template’s **`build_context()`**, run **`transform()`** on the assembled `Context` (from 0.11.0, complexity-aware **verbosity** adjustment), set **`Constraints`** manually (or via **`PromptArchitect`**), then call your provider as usual. For **PromptArchitect**-built contexts, install **≥ 0.11.1** so `assemble()` folds architect-suggested quality fields into **## GUARD RAILS**.

## Step 5 — `smart_generic_prompt`: instant prompt, zero compile cost

When you need a prompt fast and don't need full template compilation, `smart_generic_prompt` fills the GENERIC_PROMPT slot for the best-matching template. Heuristic-only, sub-millisecond.

In [12]:
from mycontext.intelligence import smart_generic_prompt

result = smart_generic_prompt(
    "Analyze the competitive positioning of our product against three alternatives",
    provider=None,  # heuristic routing, no API key needed
)

md(f"Template selected: {getattr(result, 'template_name', 'N/A')}")
prompt_str = result.to_string() if hasattr(result, 'to_string') else str(result)
md("\nGeneric prompt preview:")
md((str(prompt_str)[:600-3] + '...' if len(str(prompt_str)) > 600 else str(prompt_str)))


Template selected: N/A


Generic prompt preview:

Answer this question thoroughly and provide actionable recommendations:

Analyze the competitive positioning of our product against three alternatives

## The proof: same question, three paths

Let's see the same question through all three lenses — raw, single-template, and smart — and compare.

In [ ]:
from mycontext.intelligence import QualityMetrics, transform
from mycontext import Context, Guidance, Directive

question = "What risks should we consider before launching an AI feature in a regulated industry?"

# Path A: raw prompt (what most people do)
raw_ctx = Context(
    guidance=Guidance(role="assistant", rules=["Be helpful and accurate"]),
    directive=Directive(content=question),
)

# Path B: smart template selection
smart_ctx = transform(question)

qm = QualityMetrics(mode="heuristic")
raw_score = qm.evaluate(raw_ctx)
smart_score = qm.evaluate(smart_ctx)

md(f"{'Path':<30} {'Quality Score':>15} {'Dimensions'}")
md("-" * 70)
md(f"{'Raw (generic assistant)':<30} {raw_score.overall * 100:>14.0f}/100")
md(f"{'smart_execute (auto-routed)':<30} {smart_score.overall * 100:>14.0f}/100")
md(f"\nLift: +{(smart_score.overall - raw_score.overall) * 100:.0f} points from automatic template selection")


## Summary

| What you learned | API |
|-----------------|-----|
| Zero-decision execution | `smart_execute(question, provider=...)` |
| Inspect prompt before spending tokens | `smart_prompt(question)` |
| Fast heuristic-only prompt | `smart_generic_prompt(question)` |
| Understand routing logic | `assess_complexity(question)` |
| Pre-flight quality check | `QualityMetrics(mode="heuristic").evaluate(ctx)` |

**Next notebook:** [02_prompt_architect.ipynb](02_prompt_architect.ipynb) — when you want to build, parse, or improve a prompt yourself rather than letting `smart_execute` choose for you.